[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sanjaynagi/anopheles-omics-training/blob/main/day-3/3-1-cnvs-and-snps.ipynb)

# 3.1 Why is my gene overexpressed?

**Theme:** Theory | Analysis

On Day 2 you found genes that are differentially expressed in the Busia colony, and in Module 2.3
you checked whether they are overexpressed in other studies. Expression data tell us *that* a gene
is overexpressed, but not *why*. Today we move from the transcriptome to the genome.

In this module we ask which genetic changes could explain overexpression of your candidate genes,
and look for two of them in wild mosquitoes collected around Busia: **copy number variants (CNVs)**
and **amino acid changes**. We use whole genomes from the MalariaGEN *Ag3* resource for Ugandan
and Kenyan *An. gambiae* collected between 2012 and 2019, including the Busia 2016 cohort that we
will meet again in the selection atlas (Module 3.2).

You already know how to compute CNV and SNP frequencies with `malariagen_data` from the
MalariaGEN–PAMCA course ([W2-M4](https://anopheles-genomic-surveillance.github.io/workshop-2/module-4-cnv-frequencies.html),
[W1-M4](https://anopheles-genomic-surveillance.github.io/workshop-1/module-4-vgsc-snps.html) and
[W7-M1](https://anopheles-genomic-surveillance.github.io/workshop-7/module-1-op-ir.html)).
Here we use those tools to build evidence for (or against) a mechanism.

## Learning objectives

At the end of this module you will be able to:

- Explain three ways a gene can become overexpressed (gene amplification, *cis*-regulatory change, *trans*-acting change) and why coding changes are a separate mechanism.
- Say which kinds of evidence tell these mechanisms apart, and which of them wild-caught genome data can provide.
- Compute gene CNV frequencies and named CNV allele frequencies for your genes in Ugandan and Kenyan cohorts through time.
- Compute amino acid substitution frequencies and combine them with CNV frequencies.
- Combine population genomic evidence with AnoExpress fold changes into a short verdict for your gene dossier.

## Setup

We pin the package versions used to write this module. Accessing *Ag3* data needs a Google
account; follow the [data access instructions](https://malariagen.github.io/vector-data/vobs/vobs-data-access.html)
when you run `Ag3()` for the first time.

In [1]:
%pip install -q --no-warn-conflicts malariagen_data==15.9.0 anoexpress==0.3.3

In [2]:
import sys
import os
os.environ["MGEN_SHOW_PROGRESS"] = "0"  # hide progress bars; set to "1" to see them
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import malariagen_data
import anoexpress as xpe
pio.templates.default = "simple_white"  # course style for every Plotly figure

# Colab needs its own renderer; elsewhere, save figures as Plotly JSON so the book can show them
pio.renderers.default = "colab" if "google.colab" in sys.modules else "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", 50)

In [3]:
ag3 = malariagen_data.Ag3()
ag3

<MalariaGEN Ag3 API client>
Storage URL                           : gs://vo_agam_release_master_us_central1/
Data releases available               : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
Results cache                         : None
Cohorts analysis                      : 20260120
AIM analysis                          : 20220528
Site filters analysis                 : dt_20200416
Software version                      : malariagen_data 15.9.0
Client location                       : England, United Kingdom
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
---
Please note that data are subject to terms of use,
for more information see the Vector Observatory website https://www.malariagen.net/vobs/
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.9.0/Ag3.html

In [4]:
# Replace with the genes you were assigned on Day 2 (see data/candidate-genes.tsv)
CANDIDATE_GENES = ["AGAP002862", "AGAP009193"]  # CYP6AA1, GSTE4

In [5]:
df_genes = ag3.genome_features(attributes=["ID", "Name", "description"]).query(
    "type == 'gene' and ID in @CANDIDATE_GENES"
)
df_genes[["ID", "Name", "contig", "start", "end", "strand", "description"]]

,ID,Name,contig,start,end,strand,description
73275,AGAP002862,CYP6AA1,2R,28480576,28482637,-,cytochrome P450 [Source:VB Community Annotation]
155971,AGAP009193,GSTE4,3R,28595948,28596868,-,glutathione S-transferase epsilon class 4 [Sou...


## Theory: how does a gene become overexpressed?

A gene is transcribed more when there are more copies of it, when its own regulatory DNA
changes, or when the proteins that regulate it change. Changes to the protein itself are a
fourth, separate route to resistance.

**1. Gene amplification (CNV).** A duplication copies the gene (often with its neighbours) so the
genome carries three, four or more copies. More templates usually means more transcript.
Metabolic resistance genes in *Anopheles* are frequently amplified: Lucas *et al.* [1] found CNVs
at the *Cyp6aa/Cyp6p*, *Cyp6m/z*, *Cyp9k1* and *Gste* clusters across Africa, and a duplication
spanning *Cyp6aa1* (`Cyp6aap_Dup1`) has spread across East Africa together with other resistance
mutations [2].

**2. *Cis*-regulatory change.** A SNP, indel or transposable element insertion in the promoter,
enhancer or UTR of the gene changes how strongly *that copy* is transcribed. The best-understood
example in malaria vectors is in *An. funestus*, where promoter variants and an insertion
upstream of *CYP6P9a/b* drive overexpression [3, 4].

**3. *Trans*-acting change.** A change in a transcription factor or signalling pathway raises
the expression of many genes at once, wherever they are in the genome. The Maf-S / cnc / Keap1
pathway regulates many detoxification genes in *An. gambiae* [5], and the selection atlas flags a
sweep at *Keap1* (Module 3.2).

**4. Coding changes (an alternative, not an explanation).** An amino acid substitution can make an
enzyme better at binding or breaking down an insecticide without changing its expression, for
example GSTE2 L119F in *An. funestus* [6]. Coding changes do not explain overexpression, but they
often sit on the same haplotype as a CNV or regulatory variant and can be useful markers of it.

Finally, remember that a fold change is a *comparison*: a gene can look up or down because of the
reference strain, rearing conditions or insecticide exposure (induction), not only because of
genetic change in the resistant population.

### How can we tell the mechanisms apart?

| Evidence | CNV | *cis* | *trans* | Can we get it from wild-caught WGS? |
|---|---|---|---|---|
| Higher sequencing coverage over the gene; discordant reads at breakpoints | yes | no | no | **yes** (`gene_cnv`, `cnv_discordant_read_calls`) |
| Expression rises with copy number across individuals | yes | no | no | needs RNA and DNA from the same individuals |
| Allele-specific expression in heterozygotes | no (usually) | **yes** | no | needs RNA from heterozygotes |
| Neighbouring genes up together | often (co-amplified) | sometimes | only if co-regulated | partly (look at neighbours in the CNV) |
| Many unlinked genes up together | no | no | **yes** | no (use AnoExpress co-expression, Module 2.3) |
| Selection signal at the gene | yes | yes | no, it sits at the regulator | **yes** (Module 3.2) |
| Candidate variants at high frequency near the gene | CNV alleles | promoter SNPs / insertions | variants at the regulator | **yes** (SNP and CNV frequencies) |

So with *Ag3* we can test the CNV hypothesis directly, find coding changes, and find variants
that rise in frequency near the gene. We cannot prove a *cis* or *trans* mechanism with genomes
alone, but we can say which explanations the data support.

## The cohorts: Busia and its neighbours

Busia (Uganda) sits on the Kenyan border. *Ag3* has *An. gambiae* from Busia in 2016, 2017 and 2019,
from neighbouring districts (Namayingo, Mayuge, Tororo), from Teso North in Busia County on the
Kenyan side (2019), and from Kanungu in western Uganda (2012) for contrast. Siaya itself (Day 4) is
not in *Ag3*; Teso North is the closest Kenyan cohort.

In [6]:
SAMPLE_SETS = [
    "AG1000G-UG",                     # Tororo and Kanungu, 2012
    "1178-VO-UG-LAWNICZAK-VMF00025",  # Busia 2016-17, Mayuge 2017
    "1288-VO-UG-DONNELLY-VMF00168",   # Namayingo 2017-19
    "1288-VO-UG-DONNELLY-VMF00219",   # Busia 2017 and 2019
    "1274-VO-KE-KAMAU-VMF00246",      # Teso North (Busia County, Kenya) 2019
]
DISTRICTS = ["Busia", "Namayingo", "Mayuge", "Tororo", "Teso North", "Kanungu"]
SAMPLE_QUERY = f"taxon == 'gambiae' and admin2_name in {DISTRICTS}"
COHORTS = "admin2_year"  # group by district and year; labels look like UG-E_Busia_gamb_2016

df_samples = ag3.sample_metadata(sample_sets=SAMPLE_SETS, sample_query=SAMPLE_QUERY)
df_samples.pivot_table(
    index=["country", "admin2_name"], columns="year", values="sample_id",
    aggfunc="count", fill_value=0,
)

year                 2012  2013  2016  2017  2018  2019
country admin2_name                                    
Kenya   Teso North      0     4     0     0     0    31
Uganda  Busia           0     1    24    31     0    74
        Kanungu        95     0     0     0     0     0
        Mayuge          0     0     0    21     0     0
        Namayingo       0     0     0    19    34    11
        Tororo        112     0     0     0     0     0

Frequency functions drop cohorts with fewer than 10 mosquitoes by default (`min_cohort_size=10`),
so the single Busia 2013 mosquito and the four Teso North 2013 mosquitoes are ignored.

## CNVs at your candidate genes

`gene_cnv_frequencies()` summarises the copy number HMM calls by gene: for each cohort it reports
the fraction of mosquitoes with more copies (`amp`) or fewer copies (`del`) than expected
(see [PAMCA W2-M4](https://anopheles-genomic-surveillance.github.io/workshop-2/module-4-cnv-frequencies.html)).
We can pass gene IDs straight to `region`. The function also returns any other gene that overlaps
the region, so we keep only our candidates.

In [7]:
cnv_frq = ag3.gene_cnv_frequencies(
    region=CANDIDATE_GENES,
    cohorts=COHORTS,
    sample_sets=SAMPLE_SETS,
    sample_query=SAMPLE_QUERY,
    drop_invariant=False,
).query("gene_id in @CANDIDATE_GENES")

ag3.plot_frequencies_heatmap(cnv_frq, title="Gene CNV frequencies, candidate genes");

A CNV at your gene is a candidate explanation only if it is common in the population the RNA
came from. A CNV found in 2 % of mosquitoes cannot explain a large fold change in a colony
unless the colony happens to carry it, so always look at the frequency near Busia.

## Worked example 1: the *Cyp6aa/Cyp6p* duplication

The *Cyp6aa/Cyp6p* cluster on 2R (~28.48–28.51 Mb) holds *Cyp6aa1*, *Cyp6p3*, *Cyp6p4* and other
P450s. Let's look at gene CNV frequencies across the whole cluster.

In [8]:
CYP6_REGION = "2R:28,480,000-28,510,000"

cyp6_cnv = ag3.gene_cnv_frequencies(
    region=CYP6_REGION,
    cohorts=COHORTS,
    sample_sets=SAMPLE_SETS,
    sample_query=SAMPLE_QUERY,
)
ag3.plot_frequencies_heatmap(
    cyp6_cnv.query("max_af > 0.05"), title="Gene CNV frequencies, Cyp6aa/Cyp6p cluster"
);

Almost every mosquito from Busia, Namayingo, Mayuge and Teso North carries an amplification of
*Cyp6aa1*, whereas amplification of *Cyp6p3* is found in at most 15 % of mosquitoes in any cohort. The coverage HMM tells us *that* a gene is
amplified, not *which* duplication it is. For the known resistance loci, *Ag3* also provides
**named CNV alleles** called from discordant and split reads at the breakpoints [1], e.g.
`Cyp6aap_Dup1a`. The `Dup0` entries are an "any amplification at this locus" aggregate, not an allele.

The function below turns those calls into carrier frequencies per cohort, in the same format as
the other frequency functions so we can reuse `plot_frequencies_heatmap()`.

In [9]:
def cnv_allele_frequencies(contig, prefix, sample_query=SAMPLE_QUERY, cohorts=COHORTS,
                           min_cohort_size=10):
    # Fraction of mosquitoes in each cohort that carry each named CNV allele.
    ds = ag3.cnv_discordant_read_calls(
        contig=contig, sample_sets=SAMPLE_SETS, sample_query=sample_query
    )
    ok = ~ds["sample_is_high_variance"].values  # drop noisy samples, as gene_cnv_frequencies does
    calls = pd.DataFrame(
        ds["call_genotype"].values[:, ok].T,
        index=ds["sample_id"].values[ok],
        columns=ds["variant_id"].values,
    )
    alleles = [a for a in calls.columns if a.startswith(prefix) and not a.endswith("Dup0")]
    df_meta = ag3.sample_metadata(sample_sets=SAMPLE_SETS, sample_query=sample_query)
    cohort_col = "cohort_" + cohorts
    df = calls[alleles].join(df_meta.set_index("sample_id")[cohort_col])
    sizes = df[cohort_col].value_counts()
    df = df[df[cohort_col].isin(sizes[sizes >= min_cohort_size].index)]
    frq = (df.groupby(cohort_col)[alleles].mean().T).add_prefix("frq_")
    frq["max_af"] = frq.max(axis=1)
    frq["label"] = frq.index
    return frq.query("max_af > 0")


cyp6_alleles = cnv_allele_frequencies(contig="2R", prefix="Cyp6aap")
ag3.plot_frequencies_heatmap(cyp6_alleles, title="Cyp6aa/Cyp6p CNV alleles (carrier frequency)");

The amplification is one allele, `Cyp6aap_Dup1a`, a duplication that includes *Cyp6aa1* but not
*Cyp6p3*. It is close to fixation around Busia and lower in Tororo and Kanungu in 2012. This is
the duplication described by Njoroge *et al.* [2] as part of a "triple mutant" haplotype that
spread through East Africa.

Let's plot the carrier frequency through time in each district.

In [10]:
def tidy_allele_frequencies(frq):
    # Long format with district and year parsed from labels like UG-E_Busia_gamb_2016.
    long = (
        frq.filter(like="frq_")
        .rename(columns=lambda c: c.removeprefix("frq_"))
        .reset_index(names="allele")
        .melt(id_vars="allele", var_name="cohort", value_name="frequency")
    )
    parts = long["cohort"].str.split("_", expand=True)
    long["district"] = parts[1].str.replace("-", " ")
    long["year"] = parts[3].astype(int)
    return long


px.line(
    tidy_allele_frequencies(cyp6_alleles).query("allele == 'Cyp6aap_Dup1a'"),
    x="year", y="frequency", color="district", markers=True, range_y=[0, 1.05],
    title="Cyp6aap_Dup1a carrier frequency", width=700, height=400,
)

## Worked example 2: *Cyp9k1* amplifications

*Cyp9k1* on the X chromosome is another P450 linked to pyrethroid resistance [7] and is often
amplified [1]. On X the function accounts for males having one copy.

In [11]:
cyp9k1_cnv = ag3.gene_cnv_frequencies(
    region="AGAP000818",
    cohorts=COHORTS,
    sample_sets=SAMPLE_SETS,
    sample_query=SAMPLE_QUERY,
).query("gene_id == 'AGAP000818'")

cyp9k1_alleles = cnv_allele_frequencies(contig="X", prefix="Cyp9k1")

ag3.plot_frequencies_heatmap(
    pd.concat([cyp9k1_cnv, cyp9k1_alleles]),
    title="Cyp9k1: gene CNV (HMM) and named CNV alleles",
);

Between a third and a half of mosquitoes around Busia carry a *Cyp9k1* amplification, mostly
the allele `Cyp9k1_Dup8`. How many extra copies do they carry? `gene_cnv()` gives the modal copy
number over each gene for each mosquito.

In [12]:
BUSIA_2016 = "cohort_admin2_year == 'UG-E_Busia_gamb_2016' and sex_call == 'F'"

ds_cn = ag3.gene_cnv(region="AGAP000818", sample_sets=SAMPLE_SETS, sample_query=BUSIA_2016)
ix = list(ds_cn["gene_id"].values).index("AGAP000818")
copy_number = pd.Series(ds_cn["CN_mode"].values[ix], name="copy number")
copy_number.value_counts().sort_index().rename("n mosquitoes (Busia 2016 females)")

Females normally have 2 copies of an X-linked gene; carriers of the amplification have 3 or more.
In Module 3.3 we will see that the *Cyp9k1* amplification sits on a swept haplotype.

## Amino acid changes in your candidate genes

Next we look for common amino acid substitutions with `aa_allele_frequencies()`
([PAMCA W1-M4](https://anopheles-genomic-surveillance.github.io/workshop-1/module-4-vgsc-snps.html)).
We use each gene's canonical transcript and keep changes above 5 % in at least one cohort.

In [13]:
aa_frqs = {}
for gene in CANDIDATE_GENES:
    transcript = ag3.canonical_transcript(gene)
    df = ag3.aa_allele_frequencies(
        transcript=transcript,
        cohorts=COHORTS,
        sample_sets=SAMPLE_SETS,
        sample_query=SAMPLE_QUERY,
    )
    aa_frqs[gene] = df.query("max_af > 0.05")
    print(f"{gene} ({transcript}): {len(aa_frqs[gene])} amino acid changes above 5 %")

AGAP002862 (AGAP002862-RA): 17 amino acid changes above 5 %
AGAP009193 (AGAP009193-RA): 12 amino acid changes above 5 %


In [14]:
for gene, df in aa_frqs.items():
    if len(df) > 0:
        ag3.plot_frequencies_heatmap(df, title=f"Amino acid frequencies, {gene}")

Amino acid changes that are close to fixation in every cohort (including Kanungu, far from
Busia) are usually differences between the reference genome (the PEST strain) and East African
*An. gambiae*, not resistance mutations. Look for changes that vary between places or times.

### Combining SNPs and CNVs at the *Cyp6* cluster

At the *Cyp6* cluster, CYP6P4 I236M is part of the East African "triple mutant" together with
`Cyp6aap_Dup1` and a transposon insertion upstream of *Cyp6p4* [2]. Following the pattern from
[PAMCA W7-M1](https://anopheles-genomic-surveillance.github.io/workshop-7/module-1-op-ir.html),
we put SNP and CNV frequencies on one heatmap.

In [15]:
cyp6p4_aa = ag3.aa_allele_frequencies(
    transcript="AGAP002867-RA",  # CYP6P4
    cohorts=COHORTS,
    sample_sets=SAMPLE_SETS,
    sample_query=SAMPLE_QUERY,
).query("max_af > 0.2")

cyp6_combined = pd.concat([cyp6p4_aa, cyp6_cnv.query("max_af > 0.2"), cyp6_alleles.query("max_af > 0.2")])
ag3.plot_frequencies_heatmap(cyp6_combined, title="Cyp6 cluster: amino acid, gene CNV and CNV allele frequencies");

Where `Cyp6aap_Dup1a` is common, CYP6P4 I236M is common too, which is what we expect if they
travel on the same haplotype. But in Kanungu I236M is at about 80 % while the duplication is
carried by only a quarter of mosquitoes, so I236M must also occur on haplotypes *without* the
duplication. (SNP genotypes called inside an amplified region can be biased, see W7-M1; we test
the linkage directly in Exercise 2, and with haplotypes in Module 3.3.)

Time series make the trend easier to see. The `*_advanced` functions group by area and period
and add confidence intervals.

In [16]:
ds_cyp6p4 = ag3.aa_allele_frequencies_advanced(
    transcript="AGAP002867-RA",
    area_by="admin2_name",
    period_by="year",
    sample_sets=SAMPLE_SETS,
    sample_query=SAMPLE_QUERY,
    variant_query="max_af > 0.5",
)
ag3.plot_frequencies_time_series(ds_cyp6p4, height=600, width=900, title="CYP6P4 I236M");

## Bringing in the expression data

Finally, we put the genomic evidence next to the expression evidence from AnoExpress (Module 2.3).
We load log2 fold changes and adjusted p-values for your candidate genes and the three worked
examples in the *An. gambiae*/*An. coluzzii* analysis. The Busia contrast in AnoExpress is
`BusiaSurvivors_v_Kisumu` (deltamethrin-selected Busia colony survivors vs the susceptible Kisumu strain).

In [17]:
EXAMPLE_GENES = ["AGAP002862", "AGAP002865", "AGAP000818"]  # CYP6AA1, CYP6P3, CYP9K1
BUSIA_CONTRAST = "BusiaSurvivors_v_Kisumu"
genes = list(dict.fromkeys(CANDIDATE_GENES + EXAMPLE_GENES))

fc = xpe.data(data_type="fcs", analysis="gamb_colu", gene_id=genes)
pv = xpe.data(data_type="pvals", analysis="gamb_colu", gene_id=genes)

names = (
    ag3.genome_features(attributes=["ID", "Name"])
    .query("type == 'gene'")
    .drop_duplicates("ID")
    .set_index("ID")["Name"]
)
up = (fc > 0) & (pv < 0.05)
expression = pd.DataFrame({
    "name": names.reindex(fc.index),
    "busia_log2fc": fc[BUSIA_CONTRAST].round(2),
    "busia_padj": pv[BUSIA_CONTRAST].round(3),
    "median_log2fc_all": fc.median(axis=1).round(2),
    "n_contrasts_up": up.sum(axis=1),
    "n_contrasts": fc.notna().sum(axis=1),
})
expression

,name,busia_log2fc,busia_padj,median_log2fc_all,n_contrasts_up,n_contrasts
GeneID,,,,,,
AGAP002865,CYP6P3,-3.38,0.000,2.87,12,18
AGAP002862,CYP6AA1,-0.29,0.181,1.08,10,18
AGAP000818,CYP9K1,-0.38,0.016,2.11,14,18
AGAP009193,GSTE4,0.68,0.008,0.34,5,18


In [18]:
fc_long = fc.reset_index().melt(id_vars="GeneID", var_name="contrast", value_name="log2FC")
fc_long["name"] = fc_long["GeneID"].map(names).fillna(fc_long["GeneID"])
fc_long["study"] = np.where(fc_long["contrast"] == BUSIA_CONTRAST, "Busia", "other studies")
fig = px.strip(
    fc_long, x="name", y="log2FC", color="study", hover_data=["contrast"],
    title="AnoExpress fold changes (gamb_colu)", width=800, height=450,
)
fig.add_hline(y=0, line_dash="dot")
fig

Now we build a small evidence table: expression next to the CNV and amino acid frequencies in
Busia 2016 and 2019.

In [19]:
all_cnv = pd.concat([cnv_frq, cyp6_cnv, cyp9k1_cnv]).reset_index()
amp = (
    all_cnv.query("cnv_type == 'amp'")
    .drop_duplicates("gene_id")
    .set_index("gene_id")[["frq_UG-E_Busia_gamb_2016", "frq_UG-E_Busia_gamb_2019"]]
    .add_prefix("amp_")
)

aa_max = {
    gene: df.filter(like="frq_UG-E_Busia").max().max() if len(df) else 0.0
    for gene, df in aa_frqs.items()
}

evidence = expression.join(amp)
evidence["max_missense_frq_busia"] = pd.Series(aa_max)
evidence.round(2)

,name,busia_log2fc,busia_padj,median_log2fc_all,n_contrasts_up,n_contrasts,amp_frq_UG-E_Busia_gamb_2016,amp_frq_UG-E_Busia_gamb_2019,max_missense_frq_busia
GeneID,,,,,,,,,
AGAP002865,CYP6P3,-3.38,0.00,2.87,12,18,0.15,0.00,NaN
AGAP002862,CYP6AA1,-0.29,0.18,1.08,10,18,0.95,1.00,1.0
AGAP000818,CYP9K1,-0.38,0.02,2.11,14,18,0.35,0.45,NaN
AGAP009193,GSTE4,0.68,0.01,0.34,5,18,0.05,0.01,1.0


Read the table with care:

- CYP6AA1 is strongly overexpressed in many resistant populations, especially in West Africa,
  but it is **not** significantly different in `BusiaSurvivors_v_Kisumu` (log2FC about −0.3).
  The case for CYP6AA1 in Busia rests on other evidence: the near-fixed `Cyp6aap_Dup1a`
  duplication shown above, the East African "triple mutant" described by Njoroge *et al.* [2],
  and the selected-vs-parental comparison from our own RNA-Seq-Pop run (Day 2).
- CYP6P3 is strongly *down* in Busia survivors vs Kisumu. The Kisumu samples in that experiment
  express CYP6P3 highly (see Module 2.3): a fold change depends on the reference strain.
- A variant that is fixed in Busia cannot explain expression differences *within* Busia, and
  selection on a haplotype (Modules 3.2 and 3.3) can act on any of the mutations it carries.

For your own genes, write down which mechanism(s) the data support, which they rule out, and
what experiment would settle it (for example qPCR of copy number and expression in the same
mosquitoes, or allele-specific expression).

### Exercise 1

Look at the CNV heatmap and the amino acid tables for your own candidate genes (run the notebook
with your genes in `CANDIDATE_GENES`). For each gene:

- Is it amplified in Busia? At what frequency, and is that changing between 2016 and 2019?
- Are there common amino acid changes that vary between cohorts?
- Which mechanism(s) of overexpression are still possible after this analysis?

<details><summary>Show answer</summary>

With the default genes: *CYP6AA1* (AGAP002862) is amplified in nearly every mosquito around
Busia in every year (the `Cyp6aap_Dup1a` duplication), so a CNV is a strong candidate mechanism,
and it also carries several common amino acid changes that travel with the duplication.
*GSTE4* (AGAP009193) is in the *Gste* cluster, where amplifications are present only at low
frequency around Busia (see the `cnv_frq` heatmap for your run). A CNV in a few percent of
mosquitoes cannot explain a consistent fold change, so for GSTE4 *cis*- or *trans*-regulatory
explanations remain open. Both genes lie inside Busia selection signals (Module 3.2).

</details>

### Exercise 2

Are CYP6P4 I236M and `Cyp6aap_Dup1a` on the same haplotype? In Busia almost every mosquito carries
the duplication, so test it in Tororo 2012 (`AG1000G-UG`), where both are at intermediate frequency.
Cross-tabulate each mosquito's number of 236M alleles (SNP 2R:28,497,967 G>C) against whether it
carries `Cyp6aap_Dup1a`.

*Hint:* `ag3.snp_calls(region="2R:28497967-28497967", ...)` and `ag3.cnv_discordant_read_calls(contig="2R", ...)`.

<details><summary>Show answer</summary>

```python
query = "cohort_admin2_year == 'UG-E_Tororo_gamb_2012'"
ds_snp = ag3.snp_calls(region="2R:28497967-28497967", sample_sets="AG1000G-UG", sample_query=query)
alleles = ds_snp["variant_allele"].values[0].astype(str)  # ['G', 'C', ...]
gt = ds_snp["call_genotype"].values[0]                    # (samples, ploidy)
n_236m = pd.Series((gt == list(alleles).index("C")).sum(axis=1), index=ds_snp["sample_id"].values)

ds_cnv = ag3.cnv_discordant_read_calls(contig="2R", sample_sets="AG1000G-UG", sample_query=query)
dup = pd.Series(
    ds_cnv["call_genotype"].values[list(ds_cnv["variant_id"].values).index("Cyp6aap_Dup1a")],
    index=ds_cnv["sample_id"].values,
)
pd.crosstab(n_236m.rename("n 236M alleles"), dup.rename("Cyp6aap_Dup1a carrier"))
```

In our run: of 39 mosquitoes homozygous for 236M, 31 carry the duplication; of 14 homozygous for
236I, only 2 do; heterozygotes are split (32 of 59). The two mutations are strongly associated,
but not perfectly: many 236M haplotypes lack the duplication. This fits a stepwise history in
which the duplication arose on a haplotype that already carried I236M. So a SNP assay for I236M
tracks the triple-mutant haplotype well where it dominates (Busia), but would overestimate the
duplication elsewhere. Module 3.3 looks for SNPs that tag the duplicated haplotype more precisely.

</details>

### Exercise 3

Kanungu (western Uganda, 2012) is far from Busia. Compare *Cyp9k1* CNVs there with the Busia
cohorts. Is it the same allele? What does that suggest about how *Cyp9k1* amplifications arose?

<details><summary>Show answer</summary>

In Kanungu the *Cyp9k1* amplification is carried by more than 80 % of mosquitoes, and the main
allele is the same `Cyp9k1_Dup8` (about 70 %) that dominates around Busia (30–45 %), with some
`Cyp9k1_Dup16`. One duplication allele found 300 km apart points to the spread of a single
mutation across Uganda by gene flow, rather than independent origins. (Other alleles, such as
`Dup16` in Mayuge and Kanungu, show that more than one amplification exists.)

</details>

### Exercise 4

`Cyp6aap_Dup1a` is almost fixed in Busia, yet CYP6AA1 is not significantly overexpressed in
`BusiaSurvivors_v_Kisumu` and CYP6P3 is lower in Busia survivors than in Kisumu. Give two
explanations, and one experiment that would distinguish them.

<details><summary>Show answer</summary>

Possible explanations: (1) the Kisumu colony used in that experiment has high constitutive
expression of these genes (it was the comparator in one batch only), so the contrast is
misleading; (2) the duplication raises expression only modestly or only after exposure
(induction), and the survivors were sampled without recent exposure; (3) the duplication is
selected for another reason (for example linked coding changes such as CYP6P4 I236M); (4) the
colony, reared for many generations, may differ from wild Busia mosquitoes. Useful experiments:
qPCR of CYP6AA1 expression and copy number in the same individuals, comparison against a second
susceptible strain, or expression before and after deltamethrin exposure.

</details>

## Quiz

Check your understanding. The quiz runs in Colab and in the course book.

In [20]:
%pip install -q jupyterquiz==2.9.6.4
from jupyterquiz import display_quiz
display_quiz("https://raw.githubusercontent.com/sanjaynagi/anopheles-omics-training/main/quizzes/3-1-cnvs-and-snps.json")

## Summary

- Overexpression can come from gene amplification, *cis*-regulatory change or *trans*-acting
  change; coding changes are a separate route to resistance that often travels on the same haplotype.
- Wild-caught genomes test the CNV hypothesis directly (coverage and breakpoint calls) and reveal
  common coding and linked variants; *cis* vs *trans* needs expression data from the same individuals.
- Around Busia, `Cyp6aap_Dup1a` is near fixation and travels with CYP6P4 I236M; `Cyp9k1_Dup8` is
  carried by a third to a half of mosquitoes.
- Genomic and expression evidence must be read together, and fold changes depend on the reference strain.

## Well done!

Add the CNV and SNP results for your genes to your gene dossier. In Module 3.2 we ask whether your
genes sit inside selective sweeps in wild populations.

## References

1. Lucas ER *et al.* (2019). Whole-genome sequencing reveals high complexity of copy number variation at insecticide resistance loci in malaria mosquitoes. *Genome Research* 29:1250–1261. https://doi.org/10.1101/gr.245795.118
2. Njoroge H *et al.* (2022). Identification of a rapidly-spreading triple mutant for high-level metabolic insecticide resistance in *Anopheles gambiae* provides a real-time molecular diagnostic for antimalarial intervention deployment. *Molecular Ecology*. https://doi.org/10.1111/mec.16591
3. Mugenzi LMJ *et al.* (2019). *Cis*-regulatory CYP6P9b P450 variants associated with loss of insecticide-treated bed net efficacy against *Anopheles funestus*. *Nature Communications* 10:4652. https://doi.org/10.1038/s41467-019-12686-5
4. Weedall GD *et al.* (2019). A cytochrome P450 allele confers pyrethroid resistance on a major African malaria vector, reducing insecticide-treated bednet efficacy. *Science Translational Medicine* 11:eaat7386. https://doi.org/10.1126/scitranslmed.aat7386
5. Ingham VA *et al.* (2017). The transcription factor Maf-S regulates metabolic resistance to insecticides in the malaria vector *Anopheles gambiae*. *BMC Genomics* 18:669. https://doi.org/10.1186/s12864-017-4086-7
6. Riveron JM *et al.* (2014). A single mutation in the GSTe2 gene allows tracking of metabolically based insecticide resistance in a major malaria vector. *Genome Biology* 15:R27. https://doi.org/10.1186/gb-2014-15-2-r27
7. Vontas J *et al.* (2018). Rapid selection of a pyrethroid metabolic enzyme CYP9K1 by operational malaria control activities. *PNAS* 115:4619–4624. https://doi.org/10.1073/pnas.1719663115
8. Nagi SC *et al.* (2023). RNA-Seq-Pop: exploiting the sequence in RNA-Seq, a Snakemake workflow reveals patterns of insecticide resistance in the malaria vector *Anopheles gambiae*. *Molecular Ecology Resources* 23:1195–1206. https://doi.org/10.1111/1755-0998.13759
9. MalariaGEN Vector Observatory data and `malariagen_data` documentation: https://malariagen.github.io/vector-data/
10. AnoExpress: https://github.com/sanjaynagi/AnoExpress